# AiSynth Labs · AI DRUG guide · Notebook 3
## Structure

**Author:** Toluwanimi Odunewu (@toluwanimiso)  
**Last updated:** 2026-10-30  
**Runtime:** about 4 to 7 minutes · **GPU needed:** no  
**Projects covered:** 11 to 14 (choose a structure, read the binding site, test the shape method, shortlist)  
**Licence:** CC BY 4.0

Part of the free "Get started in AI drug discovery" guide from AiSynth Labs. Run the cells from top to bottom. Change one thing at a time and write down what moved.

### Aim of this notebook
Choose a protein structure with a reason, read what touches the drug in it, check that a shape-matching method can reproduce the real shape, and use it to shortlist molecules. You will also see what this method cannot do.

**Runs on its own:** it rebuilds the dataset from Notebook 1 in the first cell. No docking program is installed: the placement here is based on overlaying shapes, which is simpler than docking and is labelled as such.

### References
1. Berman HM et al. (2000). *The Protein Data Bank.* Nucleic Acids Res 28:235-242. Structures are downloaded from [rcsb.org](https://www.rcsb.org).
2. Cheung J et al. (2012). *Structures of human acetylcholinesterase in complex with pharmacologically important ligands.* J Med Chem 55:10282-10286 (structure 4EY7, donepezil).
3. Zdrazil B et al. (2023). *The ChEMBL Database in 2023.* Nucleic Acids Res. [doi:10.1093/nar/gkad1004](https://doi.org/10.1093/nar/gkad1004).
4. Tosco P, Stiefl N, Landrum G (2014). *Bringing the MMFF force field to the RDKit: implementation and validation.* J Cheminform 6:37. Open3DAlign: Tosco P, Balle T, Shiri F (2011). J Comput Aided Mol Des 25:777-783.
5. RDKit: open-source cheminformatics, [rdkit.org](https://www.rdkit.org).

---
# Practical

In [ ]:
# --- SETUP: runs on Colab, Kaggle or on your own computer -------------------------------
import os, sys, random, importlib.util
ON_COLAB = 'google.colab' in sys.modules
ON_KAGGLE = os.path.exists('/kaggle/input')
if (ON_COLAB or ON_KAGGLE) and importlib.util.find_spec('rdkit') is None:
    print('Installing RDKit (about a minute) ...')
    !pip install -q rdkit
SEED = 42                      # fixed seed: you and everyone else should get the same numbers
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
print(f'Colab: {ON_COLAB} | Kaggle: {ON_KAGGLE} | seed {SEED}')

In [ ]:
# --- IMPORTS, SETTINGS AND THE READY LIBRARY (the same steps as Notebook 1, projects 3 and 4) --------------
import json, time, urllib.parse, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.ML.Cluster import Butina
RDLogger.DisableLog('rdApp.*')

# >>> Change these lines to work on another protein. CDK5 is CHEMBL4036, BACE1 is CHEMBL4822.
TARGET_NAME = 'ACHE'
TARGET_CHEMBL_ID = 'CHEMBL220'          # human acetylcholinesterase
ACTIVE_CUTOFF = 6.3                     # pIC50 at or above this counts as "active". It is a choice: say yours when you post.
PIC50_RANGE = (2, 12)                   # outside this range (IC50 below 1 picomolar or above 10 millimolar) a value is treated as an error and dropped
results = {}                            # every number you will want to post is saved here

CHEMBL = 'https://www.ebi.ac.uk/chembl/api/data/'

def get_json(url, tries=4):
    for k in range(tries):
        try:
            return json.load(urllib.request.urlopen(url, timeout=90))
        except Exception:
            time.sleep(2 + 3 * k)
    raise RuntimeError('could not reach ' + url)

def fetch_ic50(chembl_id):
    rows, offset = [], 0
    while True:
        q = urllib.parse.urlencode({'target_chembl_id': chembl_id, 'standard_type': 'IC50', 'standard_relation': '=',
                                    'assay_type': 'B', 'standard_units': 'nM', 'limit': 1000, 'offset': offset})
        page = get_json(CHEMBL + 'activity.json?' + q)
        rows += page['activities']
        if not page['page_meta']['next']:
            return pd.DataFrame(rows)
        offset += 1000

def rule_of_five_ok(m):
    ok = [Descriptors.ExactMolWt(m) <= 500, Lipinski.NumHAcceptors(m) <= 10, Lipinski.NumHDonors(m) <= 5, Descriptors.MolLogP(m) <= 5]
    return sum(ok) >= 3

def catalog(name):
    p = FilterCatalogParams(); p.AddCatalog(getattr(FilterCatalogParams.FilterCatalogs, name)); return FilterCatalog(p)
PAINS, BRENK = catalog('PAINS'), catalog('BRENK')

def build_library(chembl_id, apply_filters=True):
    """ChEMBL IC50 records -> one row per molecule (median pIC50) -> rule of five and PAINS filters."""
    raw = fetch_ic50(chembl_id)
    d = raw[['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'document_year']].copy()
    d['standard_value'] = pd.to_numeric(d['standard_value'], errors='coerce')
    d = d.dropna(subset=['standard_value', 'canonical_smiles']); d = d[d['standard_value'] > 0]
    d['pIC50'] = 9 - np.log10(d['standard_value'])
    d = (d.groupby('molecule_chembl_id', as_index=False)
           .agg(smiles=('canonical_smiles', 'first'), pIC50=('pIC50', 'median'), document_year=('document_year', 'min')))
    d = d[d.pIC50.between(*PIC50_RANGE)]                  # drop the impossible potencies (see Notebook 1, project 3)
    d['mol'] = [Chem.MolFromSmiles(s) for s in d['smiles']]
    d = d[d['mol'].notna()].reset_index(drop=True)
    n_all = len(d)
    if apply_filters:
        d = d[[rule_of_five_ok(m) for m in d.mol]]
        d = d[[not PAINS.HasMatch(m) for m in d.mol]]
    return d.reset_index(drop=True), n_all

lib, n_all = build_library(TARGET_CHEMBL_ID)
print(f'{TARGET_NAME} ({TARGET_CHEMBL_ID}): {n_all:,} molecules in ChEMBL, {len(lib):,} left after the rule of five and PAINS')
results['library'] = dict(target=TARGET_NAME, molecules_all=n_all, molecules_library=len(lib))

## Project 11. Choose the structure
The Protein Data Bank holds experimental 3D structures. We ask for X-ray structures of the protein at 3.0 A resolution or better (lower numbers are sharper) and rank them. A good pick has a drug-like ligand in the pocket you care about. Several chains in one entry are normal.

In [ ]:
# --- PROJECT 11: choose the structure ------------------------------------------------------------------------
UNIPROT = {'CHEMBL220': 'P22303', 'CHEMBL4036': 'Q00535', 'CHEMBL4822': 'P56817'}[TARGET_CHEMBL_ID]     # the protein's UniProt ID
ADDITIVES = {'NAG', 'FUC', 'BMA', 'MAN', 'EDO', 'GOL', 'SO4', 'PEG', 'PGE', 'ACT', 'CL', 'NA', 'PO4', 'DMS', 'TRS', 'ZN', 'CA', 'MG'}

def rcsb_search(uniprot, max_resolution=3.0, rows=60):
    """X-ray structures of one protein, best resolution first (RCSB search API)."""
    q = {'query': {'type': 'group', 'logical_operator': 'and', 'nodes': [
            {'type': 'terminal', 'service': 'text', 'parameters': {'attribute': 'rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession', 'operator': 'exact_match', 'value': uniprot}},
            {'type': 'terminal', 'service': 'text', 'parameters': {'attribute': 'exptl.method', 'operator': 'exact_match', 'value': 'X-RAY DIFFRACTION'}},
            {'type': 'terminal', 'service': 'text', 'parameters': {'attribute': 'rcsb_entry_info.resolution_combined', 'operator': 'less_or_equal', 'value': max_resolution}}]},
         'return_type': 'entry', 'request_options': {'paginate': {'start': 0, 'rows': rows}, 'sort': [{'sort_by': 'rcsb_entry_info.resolution_combined', 'direction': 'asc'}]}}
    req = urllib.request.Request('https://search.rcsb.org/rcsbsearch/v2/query', data=json.dumps(q).encode(), headers={'Content-Type': 'application/json'})
    r = json.load(urllib.request.urlopen(req, timeout=60))
    return r['total_count'], [x['identifier'] for x in r['result_set']]

total, ids = rcsb_search(UNIPROT)
rows = []
for pid in ids[:25]:
    e = get_json(f'https://data.rcsb.org/rest/v1/core/entry/{pid}')
    info = e['rcsb_entry_info']
    ligs = [c for c in (info.get('nonpolymer_bound_components') or []) if c not in ADDITIVES]
    rows.append(dict(pdb_id=pid, resolution=info['resolution_combined'][0], chains=info.get('deposited_polymer_entity_instance_count'), ligands=','.join(ligs) or '-', title=e['struct']['title'][:60]))
cand = pd.DataFrame(rows)
print(f'{total} X-ray structures of {TARGET_NAME} ({UNIPROT}) at 3.0 A or better. The best 25 by resolution:')
display(cand)
results['p11'] = dict(uniprot=UNIPROT, n_structures=total, best=cand.head(3)[['pdb_id', 'resolution']].to_dict('records'))

# Your choice. A good structure has a drug-like ligand in the pocket you care about. For acetylcholinesterase, 4EY7 holds donepezil.
PDB_ID, LIGAND_ID, LIGAND_CHAIN = '4EY7', 'E20', 'A'        # E20 is donepezil's three-letter code in the PDB
chosen = get_json(f'https://data.rcsb.org/rest/v1/core/entry/{PDB_ID}')
print('chosen:', PDB_ID, chosen['struct']['title'], '| resolution', chosen['rcsb_entry_info']['resolution_combined'][0], 'A')
results['p11']['chosen'] = dict(pdb_id=PDB_ID, resolution=chosen['rcsb_entry_info']['resolution_combined'][0], title=chosen['struct']['title'])

## Project 12. Read the binding site
We list which protein residues have an atom within 4 A of the ligand, which atom pairs could be hydrogen bonds (nitrogen or oxygen within 3.5 A of each other) and how many carbon-carbon contacts each residue makes. These are geometric rules, not a full interaction analysis. Your checklist is the three to five residues a good ligand should keep touching.

In [ ]:
# --- PROJECT 12: read the binding site ---------------------------------------------------------------------------
pdb_lines = urllib.request.urlopen(f'https://files.rcsb.org/download/{PDB_ID}.pdb', timeout=90).read().decode().splitlines()

def parse(lines, record):
    out = []
    for l in lines:
        if l.startswith(record):
            out.append(dict(name=l[12:16].strip(), res=l[17:20].strip(), chain=l[21], resi=int(l[22:26]),
                            elem=(l[76:78].strip() or l[12:16].strip()[0]).upper(), xyz=(float(l[30:38]), float(l[38:46]), float(l[46:54]))))
    return out

prot = [a for a in parse(pdb_lines, 'ATOM') if a['elem'] != 'H']
lig = [a for a in parse(pdb_lines, 'HETATM') if a['res'] == LIGAND_ID and a['chain'] == LIGAND_CHAIN]
print(f'ligand {LIGAND_ID} chain {LIGAND_CHAIN}: {len(lig)} atoms; protein: {len(prot):,} atoms')
PXYZ, LXYZ = np.array([a['xyz'] for a in prot]), np.array([a['xyz'] for a in lig])
D = np.linalg.norm(LXYZ[:, None, :] - PXYZ[None, :, :], axis=2)            # every ligand atom to every protein atom

contacts = {}
for j in np.where(D.min(axis=0) <= 4.0)[0]:                                  # protein atoms within 4 A of the ligand
    a = prot[j]; key = (a['res'], a['chain'], a['resi'])
    c = contacts.setdefault(key, dict(residue=f"{a['res']}{a['resi']}", chain=a['chain'], atoms=0, closest=9.9, hbond=False, hydrophobic=0))
    i = int(D[:, j].argmin()); c['atoms'] += 1; c['closest'] = min(c['closest'], float(D[i, j]))
    for i2 in np.where(D[:, j] <= 4.0)[0]:
        if lig[i2]['elem'] in 'NO' and a['elem'] in 'NO' and D[i2, j] <= 3.5: c['hbond'] = True        # a possible hydrogen bond
        if lig[i2]['elem'] == 'C' and a['elem'] == 'C': c['hydrophobic'] += 1                           # carbon touching carbon
table = pd.DataFrame(contacts.values()).sort_values(['hydrophobic', 'atoms'], ascending=False).reset_index(drop=True)
table['aromatic'] = table.residue.str[:3].isin(['TRP', 'TYR', 'PHE', 'HIS'])
table['closest'] = table.closest.round(2)
print(f'{len(table)} residues have an atom within 4 A of the ligand. The ten with the most carbon contacts:')
display(table.head(10))
checklist = list(table.head(5).residue)                                      # your checklist: change this list after you have looked at the picture
print('checklist for project 14:', checklist)
results['p12'] = dict(residues_within_4A=len(table), hbond_candidates=table[table.hbond].residue.tolist(), aromatic_in_contact=table[table.aromatic].residue.tolist(),
                      top_hydrophobic=table.head(5)[['residue', 'hydrophobic']].to_dict('records'), checklist=checklist)

## Project 13. Can my method find the right shape?
A flat structure has many 3D shapes. Before using shapes to place molecules, check that the generator can produce the real one: make 300 shapes of the drug and measure how close the best is to the crystal. Under 2 A is a common rule of thumb for a pass.

In [ ]:
# --- PROJECT 13: can my method find the right shape? -------------------------------------------------------------------
from rdkit.Chem import AllChem, rdMolAlign
DONEPEZIL = 'COC1=C(C=C2C(=C1)CC(C2=O)CC3CCN(CC3)CC4=CC=CC=C4)OC'
template = Chem.MolFromSmiles(DONEPEZIL)

# The crystal gives coordinates but no bond orders: take the bond orders from the known structure of the drug.
block = '\n'.join(l for l in pdb_lines if l.startswith('HETATM') and l[17:20].strip() == LIGAND_ID and l[21] == LIGAND_CHAIN) + '\nEND\n'
crystal = AllChem.AssignBondOrdersFromTemplate(template, Chem.MolFromPDBBlock(block, proximityBonding=True))
print('crystal ligand heavy atoms:', crystal.GetNumAtoms(), '| rotatable bonds:', Descriptors.NumRotatableBonds(crystal))

# Generate 300 shapes from the flat structure, then ask: does any of them match the crystal shape?
mol = Chem.AddHs(template)
cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=300, randomSeed=SEED))
AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=500)
heavy = Chem.RemoveHs(mol)
rms = []
for cid in cids:
    probe = Chem.Mol(heavy); probe.RemoveAllConformers(); probe.AddConformer(heavy.GetConformer(cid), assignId=True)
    rms.append(rdMolAlign.GetBestRMS(probe, Chem.Mol(crystal)))               # best match after rotating and moving, counting symmetry
rms = np.array(rms)
for n in (10, 50, 100, 300):
    print(f'  best heavy-atom RMSD among the first {n:>3} shapes: {rms[:n].min():.2f} A')
print(f'shapes within 1.0 A: {(rms < 1).sum()} | within 2.0 A: {(rms < 2).sum()} of {len(rms)}')
results['p13'] = dict(conformers=len(rms), best_rmsd=round(float(rms.min()), 2), best_first_10=round(float(rms[:10].min()), 2), best_first_50=round(float(rms[:50].min()), 2),
                      within_1A=int((rms < 1).sum()), within_2A=int((rms < 2).sum()), rotatable_bonds=int(Descriptors.NumRotatableBonds(crystal)))
# Rule of thumb: if the best RMSD is under 2 A the shape generator can find the real shape. If not, do not trust a placement built on it.

## Project 14. A structure-supported shortlist
We overlay each candidate on the crystal ligand (shape and chemistry together), then count clashes with the protein and measure how many contacts it makes with your checklist residues compared with the crystal ligand itself (ratio 1.0 = the same). Donepezil itself is the positive control (it should score highest and set the baseline for clashes) and metformin is the negative control (it should score low and touch little). **This is a quick filter, not docking**: nothing here searches for new poses.

In [ ]:
# --- PROJECT 14: a structure-supported shortlist (template-based placement, NOT docking) ------------------------------
# Idea: lay each candidate over the crystal ligand by overall shape and chemistry, then ask two questions:
# does it bump into the protein (clashes), and does it still touch the residues on your checklist?
from rdkit.Chem.Scaffolds import MurckoScaffold
crystal_h = Chem.AddHs(crystal, addCoords=True)
mp_ref = AllChem.MMFFGetMoleculeProperties(crystal_h)
fp2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
f_don = fp2.GetFingerprint(template)
lib['sim_to_donepezil'] = DataStructs.BulkTanimotoSimilarity(f_don, [fp2.GetFingerprint(m) for m in lib.mol])
band = lib[(lib.pIC50 >= 7.5) & (lib.sim_to_donepezil.between(0.3, 0.8))].sort_values('pIC50', ascending=False)
picked, seen = [], set()
for r in band.itertuples():                                                   # ten potent molecules with different scaffolds
    sc = Chem.MolToSmiles(MurckoScaffold.MakeScaffoldGeneric(MurckoScaffold.GetScaffoldForMol(r.mol)))
    if sc not in seen:
        seen.add(sc); picked.append((r.molecule_chembl_id, r.mol, r.pIC50))
    if len(picked) == 10: break
controls = [('donepezil (positive control)', template, np.nan), ('metformin (negative control)', Chem.MolFromSmiles('CN(C)C(=N)N=C(N)N'), np.nan)]

check_idx = [j for j, a in enumerate(prot) if f"{a['res']}{a['resi']}" in checklist and a['chain'] == LIGAND_CHAIN]
ref_contacts = int((np.linalg.norm(LXYZ[:, None, :] - PXYZ[None, check_idx, :], axis=2) <= 4.0).sum())     # the crystal ligand's own contacts with the checklist
def place(name, m0, pic, n_conf=60):
    m = Chem.AddHs(m0)
    cids = list(AllChem.EmbedMultipleConfs(m, numConfs=n_conf, randomSeed=SEED))
    if not cids: return None
    AllChem.MMFFOptimizeMoleculeConfs(m, maxIters=300)
    mp = AllChem.MMFFGetMoleculeProperties(m)
    best = None
    for cid in cids:
        o3a = rdMolAlign.GetO3A(m, crystal_h, prbPyMMFFMolProperties=mp, refPyMMFFMolProperties=mp_ref, prbCid=cid)
        o3a.Align()                                                           # moves this shape into the protein's coordinate frame
        s = o3a.Score()
        if best is None or s > best[0]:
            xyz = np.array([m.GetConformer(cid).GetAtomPosition(a.GetIdx()) for a in m.GetAtoms() if a.GetAtomicNum() > 1])
            best = (s, xyz)
    dd = np.linalg.norm(best[1][:, None, :] - PXYZ[None, :, :], axis=2)
    clashes = int((dd < 3.0).sum())                                           # heavy atoms closer than 3.0 A to the protein
    ratio = float((dd[:, check_idx] <= 4.0).sum()) / ref_contacts             # 1.0 = as many checklist contacts as the crystal ligand
    return dict(molecule=name, pIC50=None if np.isnan(pic) else round(float(pic), 2), shape_score=round(float(best[0]), 1), clashes=clashes,
                checklist_contact_ratio=round(ratio, 2))

rows = [place(n, m0, p) for n, m0, p in controls + picked]
shortlist = pd.DataFrame([r for r in rows if r]).sort_values('shape_score', ascending=False)
display(shortlist)
results['p14'] = dict(rows=shortlist.to_dict('records'), checklist=checklist)

---
# Your turn
**Task:** pick a different structure from the project 11 table (a different ligand, or CDK5 via `TARGET_CHEMBL_ID = 'CHEMBL4036'` and its own `PDB_ID`, `LIGAND_ID`, `LIGAND_CHAIN`), and rerun projects 12 and 13.

**Expected output:** a different checklist, and a best shape RMSD you can compare with the one above.

In [ ]:
print(json.dumps(results, indent=1, default=str)[:3500])

---
# Discussion
**What you learned:** a structure is an argument, not a fact: resolution, ligand and chain choices all shape what you read from it.

**What this cannot do, and where it breaks:** distance rules find candidates for hydrogen bonds and hydrophobic contacts, they do not prove them. Overlaying on one crystal ligand assumes the candidate binds the same way, which is wrong for molecules with a different binding mode. A shape score is not a binding energy. The controls matter: if the positive control does not fit or the negative control does, distrust the whole table.